In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter


def prepare_docs():
    # =============1.准备文档================
    with open('./resources/评估.md', 'r', encoding='utf-8') as f:
        doc = f.read()

    # =============2.切分文档================
    # 切分依据，这里是按照三级标题
    headers_to_split_on = [
        ("#", "h1"),
        ("##", "h2"),
        ("###", "h3"),
    ]

    # 创建切分器
    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on,
        strip_headers=False  # 在正文中是否去除标题，False，不去除
    )

    # 切分文档, 返回的是 list[Document]
    docs = markdown_splitter.split_text(doc)

    for i, doc in enumerate(docs, start=1):
        doc.id = f"{i}"

    return docs


docs = prepare_docs()

from langchain_ollama import OllamaEmbeddings

# 1.初始化向量模型，可以把文档转稠密向量
embeddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b",  # 性价比高的模型
    dimensions=1024  # 可选：减少维度以节省存储
)

from pymilvus import MilvusClient
from app.core.config import settings

# 初始化客户端
client = MilvusClient(uri=settings.rag.milvus_url)

# Collection名称
collection_name = "my_collection_1"



In [ ]:
from pymilvus import Function, FunctionType, DataType


def create_collection():
    # 1.schema
    schema = client.create_schema(
        auto_id=False,  # 不自动生成id
        enable_dynamic_field=False,  # 是否允许动态字段
    )

    # 1.1.主键字段
    schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
    # 1.2.稠密向量字段
    schema.add_field(field_name="dense", datatype=DataType.FLOAT_VECTOR, dim=1024)
    # 1.3.稀疏向量字段
    schema.add_field(field_name="sparse", datatype=DataType.SPARSE_FLOAT_VECTOR)
    # 1.4.普通元数据字段，用来做数据过滤
    schema.add_field(field_name="h2", datatype=DataType.VARCHAR, max_length=128)
    # 1.5.原始文档内容，需要被向量化
    schema.add_field(
        field_name="content",
        datatype=DataType.VARCHAR,
        max_length=2048,
        enable_analyzer=True,  # 允许对该字段分词，转BM25稀疏向量
        analyzer_params={"type": "chinese"}  # 指定分词方式，这里选择中文分词
    )
    # 1.6.添加BM25函数，可以自动把原始文档转BM25向量存储
    bm25_function = Function(
        name="text_bm25_emb",  # Function name
        input_field_names=["content"],  # 包含原始文档数据的字段的名称
        output_field_names=["sparse"],  # 稀疏向量字段的名称
        function_type=FunctionType.BM25,  # Set to `BM25`
    )
    schema.add_function(bm25_function)

    # 2.索引
    index_params = client.prepare_index_params()
    # 2.1.稠密向量索引
    index_params.add_index(
        field_name="dense",  # 字段名
        index_type="AUTOINDEX",  # 稠密向量索引方式
        metric_type="COSINE"  # 相似度计算方式，COSINE、IP
    )
    # 2.2.稀疏向量索引
    index_params.add_index(
        field_name="sparse",  # 字段名
        index_type="SPARSE_INVERTED_INDEX",  # 稀疏向量，采用倒排索引
        metric_type="BM25",  # 相似度计算方式，支持 BM25、IP
    )
    # 2.3.普通字段索引
    index_params.add_index(field_name="h2", index_type="AUTOINDEX")

    # 3.创建collection
    client.create_collection(
        collection_name=collection_name,
        schema=schema,
        index_params=index_params
    )


create_collection()
###Attu@123456  root/admin

In [ ]:

##  插入数据(增)
#  1.准备数据(向量和元数据)    把文档向量化，要传入list[str]
vectors = embeddings.embed_documents([doc.page_content for doc in docs])
#  2. milvus接收的是dict，所以先做数据转换
data = [
    {
        "id": i + 1,
        "content": doc.page_content,
        "h2": doc.metadata.get("h2", ""),
        "dense": vectors[i],  # 稠密向量字段
    }
    for i, doc in enumerate(docs)
]
# 插入
res = client.insert(collection_name="my_collection_1", data=data)
print(res)


In [ ]:
#改(更新,重复iD->更新内容,不存在的iD->新增一条)
data_1 = data[0]
data_1['h2'] = "测试一下"

data_2 = data[1]
data_2['id'] = 188

client.upsert(collection_name="my_collection_1", data=[data_1, data_2])

In [ ]:
# 删  , 根据id删除
res = client.delete(collection_name="my_collection_1", ids=[188])
print(res)  # {'delete_count': 1, 'cost': 0}

In [ ]:
#查询(查) 111
#稠密向量查询
def dense_search(query: str):
    query_vector = embeddings.embed_query(query)
    res = client.search(
        collection_name="my_collection_1",  # Collection名称
        anns_field="dense",  # 检索字段名（稠密）
        data=[query_vector],  # 问题向量（稠密）
        limit=3,  # 查询文档最大数量
        search_params={"metric_type": "COSINE"},  # 向量相似度算法，必须和索引时一致
        output_fields=["content", "h2"]  # 要返回的字段名，默认只返回id和score
    )
    # 3.遍历结果
    # for hits in res:
    #     for hit in hits:
    #         print(hit)


In [ ]:
##查询(查)222
#稀疏向量查询
# 定义稀疏检索函数
def sparse_search(query: str):
    # 1.直接检索，内置的hm25函数会自动把query转向量
    res = client.search(
        collection_name=collection_name,  # Collection名称
        data=[query],
        anns_field="sparse",  # 检索字段名（稀疏）
        limit=3,  # 查询文档最大数量
        output_fields=["content", "h2"]  # 要返回的字段名
    )
    # 2.打印结果
    for hits in res:
        for hit in hits:
            print(hit)


In [ ]:
# 稠密检索
# 定义一个稠密检索的函数
def dense_search(query: str):
    # 1.先把问题向量化
    query_vector = embeddings.embed_query(query)

    # 2.然后搜索：
    res = client.search(
        collection_name=collection_name,  # Collection名称
        anns_field="dense",  # 检索字段名（稠密）
        data=[query_vector],  # 问题向量（稠密）
        limit=2,  # 查询文档最大数量
        search_params={"metric_type": "COSINE"},  # 向量相似度算法，必须和索引时一致
        output_fields=["content", "h2"]  # 要返回的字段名，默认只返回id和score
    )
    # 3.遍历结果
    for hits in res:
        for hit in hits:
            print(hit)

In [ ]:
dense_search("教育的负面作用是什么")

In [ ]:
# 测试模糊的语义搜索，稀疏检索不太擅长
sparse_search("教育的负面作用是什么")

In [ ]:
# 专有名称查询：
sparse_search("给教师的建议是谁写的")

In [ ]:
# RRF 混合检索
from pymilvus import AnnSearchRequest


def hybrid_search(query: str, ranker):
    # 1.创建稠密、稀疏请求
    # 稠密
    query_dense_vector = embeddings.embed_query(query)
    dense_request = AnnSearchRequest(
        data=[query_dense_vector],
        anns_field="dense",
        param={"metric_type": "COSINE"},
        limit=9,
    )
    # 稀疏
    sparse_request = AnnSearchRequest(
        data=[query],
        anns_field="sparse",
        param={"metric_type": "BM25"},
        limit=9,
    )
    # 混合在一起
    reqs = [dense_request, sparse_request]

    # 2.设定融合策略，发送请求
    res = client.hybrid_search(
        collection_name=collection_name,
        reqs=reqs,
        ranker=ranker,
        limit=2,
        output_fields=['content', 'h2']
    )
    for hits in res:
        print("TopK results:")
        for hit in hits:
            print(hit)

In [ ]:
from pymilvus import WeightedRanker

# 定义加权融合策略
ranker = WeightedRanker(0.5, 0.5)
# 测试
hybrid_search("教育的负面作用是什么", ranker)

In [ ]:
from pymilvus import AnnSearchRequest, RRFRanker

# 定义RRF融合策略
ranker = RRFRanker(k=60)
# 测试
hybrid_search("教育的负面作用是什么", ranker)

In [ ]:
# cross-encoder rerank
from pymilvus import Function, FunctionType


def create_cross_encoder_ranker(queries: list[str]):
    return Function(
        name="dashscope_semantic_ranker",  # ranker名词，唯一即可
        input_field_names=["content"],  # 原始文档字段
        function_type=FunctionType.RERANK,  # ranker类型，这里是固定值
        params={
            "reranker": "model",  # rerank类型，基于模型rerank，也就是cross-encoder
            "provider": "ali",  # rerank模型提供者
            "model_name": "gte-rerank-v2",  # rerank模型名称
            "queries": queries,  # 查询条件
            "max_client_batch_size": 5,  # 单批次中处理的文档最大数量
        },
    )

In [ ]:
def cross_encoder_hybrid_search(query: str):
    # 初始化一个cross-encoder的reranker
    cross_encoder_ranker = create_cross_encoder_ranker([query])
    # 混合检索
    hybrid_search(query, cross_encoder_ranker)


cross_encoder_hybrid_search("教育的负面作用是什么")

In [ ]:
# RRF 混合检索
from pymilvus import AnnSearchRequest


def hybrid_search(query: str, ranker, filter_query: str = None):
    # 1.创建稠密、稀疏请求
    # 稠密
    query_dense_vector = embeddings.embed_query(query)
    dense_request = AnnSearchRequest(
        data=[query_dense_vector],
        anns_field="dense",
        param={"metric_type": "COSINE"},
        filter=filter_query,
        limit=9,
    )
    # 稀疏
    sparse_request = AnnSearchRequest(
        data=[query],
        anns_field="sparse",
        param={"metric_type": "BM25"},
        filter=filter_query,
        limit=9,
    )
    # 混合在一起
    reqs = [dense_request, sparse_request]

    # 2.设定融合策略，发送请求
    res = client.hybrid_search(
        collection_name=collection_name,
        reqs=reqs,
        ranker=ranker,
        limit=2,
        output_fields=['content', 'h2']
    )
    for hits in res:
        print("TopK results:")
        for hit in hits:
            print(hit)

In [ ]:
from pymilvus import WeightedRanker

# 定义加权融合策略
ranker = WeightedRanker(0.5, 0.5)
# 定义过滤条件
filter_query = "h2 == '第一节 教育的功能'"
# 测试
hybrid_search("教育的负面作用是什么", ranker, filter_query)